# BC06 — Gestion et pilotage du projet (notebook de démonstration)

## De quoi parle-t-on ?

Les 5 blocs précédents produisaient du **résultat** (des données, des modèles, une API). Ce
bloc-ci parle de **méthode** : comment le projet a été **cadré, planifié, testé et documenté** de
bout en bout. Ce n'est pas « juste » écrire du code — c'est le rendre **fiable, compréhensible et
transmissible**.

## Pourquoi ce bloc ?

Le référentiel RNCP demande de savoir **piloter un projet data** : traduire un besoin métier en
problème data, planifier, identifier les risques, mettre en place des tests automatisés,
documenter, et évaluer le rapport coût / bénéfice.

## Comment suivre ce notebook

1. La **problématique métier** traduite en problématique data.
2. Le **rétroplanning** (méthode agile, 4 semaines).
3. Les **tests automatisés** — exécutés en direct.
4. La **documentation**, bloc par bloc.
5. Les **risques** identifiés et leur traitement.
6. **Coûts et bénéfices** (ROI).
7. **Gouvernance des données & RGPD**.

Le détail complet est dans [`docs/gestion_projet.md`](docs/gestion_projet.md) ; ce notebook en
donne la version commentée, et exécute réellement les tests.

## 0. Préparation

In [1]:
import sys
import subprocess
from pathlib import Path

_racine = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "commun").is_dir())
_bloc = _racine / "blocs" / "bc06_gestion_projet"
print("Racine du projet :", _racine)

Racine du projet : C:\Users\rasmi\Projects\projet_rncp\oiseaux_migrateurs_npdc


## 1. La problématique métier, traduite en problématique data

Tout part d'un besoin concret. Le travail de cadrage consiste à le **reformuler** dans des termes
que des méthodes data savent traiter.

| Enjeu métier | Traduction data |
|---|---|
| Savoir *quand* les oiseaux migrateurs arrivent dans le Nord-Pas-de-Calais | **Classification binaire** : présence / absence d'une espèce, par semaine et par maille |
| Comprendre si la météo explique ces arrivées | **Corrélation** + **importance des variables** d'un modèle supervisé |
| Rendre la prévision utilisable par un non-technicien | **API** + **tableau de bord** exposant la probabilité de présence |

**Problématique retenue :** *peut-on modéliser l'arrivée des migrations à partir de variables
climatiques et de la position géographique ?* — c'est le fil rouge des 5 blocs techniques.

## 2. Le rétroplanning (méthode agile, 4 semaines)

Le projet a été découpé en **itérations hebdomadaires**, chacune livrant quelque chose de
vérifiable. Planification **à rebours** depuis la date de soutenance.

| Semaine | Bloc(s) | Livrables | Dépend de |
|---|---|---|---|
| S1 | BC01 | Acquisition GBIF + Open-Meteo, ETL, grille présence/absence | — |
| S2 | BC02 | Saisonnalité, distributions, corrélations, test χ² | BC01 |
| S3 | BC03 + BC04 | 3 modèles ML comparés + validation + segmentation ; réseau LSTM (texte) | BC01 |
| S4 | BC05 + BC06 | API, dashboard, Docker ; tests, documentation, cette note | BC03 |

**Jalons de contrôle :**

- **J1** (fin S1) : `grille_presence_hebdo.parquet` produite → feu vert pour BC02/BC03.
- **J2** (fin S3) : modèle de production figé (`pipeline_ml.pkl`) → feu vert pour BC05.
- **J3** (mi-S4) : API + dashboard fonctionnels en local → répétition de la démo.

**Ce qui sécurise le planning.** Les fichiers `donnees/traitees/` et le modèle
`modeles/pipeline_ml.pkl` sont **versionnés dans le dépôt** : un retard sur un bloc ne bloque pas
les suivants, qui repartent de la dernière version figée.

## 3. Les tests automatisés : fiabiliser le code

**Pourquoi des tests ?** Un test automatisé est un petit programme qui vérifie qu'une fonction
**fait bien ce qu'elle est censée faire** sur un cas connu — par exemple : « la zone géographique
doit avoir une latitude minimale inférieure à sa latitude maximale ». C'est un **filet de
sécurité** : le jour où une modification casse quelque chose sans qu'on s'en rende compte, un test
qui échoue nous prévient.

Ici, la suite [`tests/test_acquisition.py`](tests/test_acquisition.py) porte sur le module
d'acquisition de BC01 (création de la *bounding box*, extraction des colonnes, structure des
espèces, cohérence de la zone). La cellule ci-dessous les **exécute réellement** avec `pytest`
(c'est ce que fait `run.py`, fonction `executer_tests`).

In [2]:
resultat = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--no-header"],
    cwd=str(_bloc), capture_output=True, text=True,
)
print(resultat.stdout[-3000:])
print("Tous les tests passent." if resultat.returncode == 0
      else "Certains tests ont échoué — voir le détail ci-dessus.")

============================= test session starts =============================
collected 6 items

tests\test_acquisition.py ......                                         [100%]

============================== 6 passed in 1.10s ==============================

Tous les tests passent.


**Ce que ça nous apprend.** Les 6 tests passent. Ils sont rejoués à chaque exécution de BC06 :
c'est une garantie que le code d'acquisition continue de se comporter comme prévu, même après des
modifications ultérieures (comme le passage de la période à 2019-2024).

## 4. La documentation, bloc par bloc

Chaque dossier `blocs/bc0X_.../` contient un **`README.md` au même format** : *objectif visé, ce qui
est implémenté, où le prouver dans le code, comment le démontrer en direct, statut*. Ce découpage
répété permet à n'importe quel membre du jury d'**auditer un bloc indépendamment des autres**, sans
tout relire, et de présenter chaque bloc en ~15 minutes avec sa propre commande d'exécution
autonome. Chaque bloc technique a aussi un **notebook** pédagogique.

In [3]:
for dossier in sorted((_racine / "blocs").glob("bc0*")):
    a_readme = "README" if (dossier / "README.md").exists() else "  --  "
    notebooks = [p.name for p in dossier.glob("notebook_*.ipynb")]
    a_notebook = notebooks[0] if notebooks else "(pas de notebook)"
    print(f"  {dossier.name:<34} {a_readme}   {a_notebook}")

  bc01_infrastructure_donnees        README   notebook_bc01.ipynb
  bc02_analyse_exploratoire          README   notebook_bc02.ipynb
  bc03_machine_learning              README   notebook_bc03.ipynb
  bc04_deep_learning                 README   notebook_bc04.ipynb
  bc05_industrialisation             README   notebook_bc05.ipynb
  bc06_gestion_projet                README   notebook_bc06.ipynb


## 5. Les risques identifiés et leur traitement

Une bonne gestion de projet ne **cache pas** ses limites : elle les nomme et propose des réponses.

| Risque | Prob. | Impact | Traitement | Statut |
|---|---|---|---|---|
| API GBIF indisponible (5xx transitoires) | Élevée | Moyen | `get_avec_retry` (backoff exponentiel) + `donnees/traitees/` versionnées | **Traité** |
| Fort déséquilibre des classes (~98 % d'absences) | Certaine | Élevé | Métriques adaptées (F1, AUC-ROC, matrice de confusion) plutôt que l'accuracy ; période bornée à 2019-2024 (pas d'absences fictives) ; SMOTE identifié pour la suite | **Traité (partiel)** |
| Sur-apprentissage du modèle retenu | Moyenne | Moyen | Validation croisée 5-fold + écart train/test contrôlé (< 0,05) | **Traité** |
| Météo passée seule, peu prédictive | Moyenne | Moyen | Limite assumée ; piste : intégrer des prévisions météo | **Accepté** |
| Biais d'effort d'observation (science citoyenne) | Certaine | Moyen | Signalé explicitement ; interprétation prudente | **Accepté** |
| Déploiement cloud non réalisé | Certaine | Moyen | Fichiers de déploiement prêts (`Procfile`, `render.yaml`) + procédure documentée | **Ouvert** |
| Incompatibilité de versions au `pip install` | Faible | Faible | `requirements.txt` épinglé + `setup_venv.ps1` (chemin court pour TensorFlow) ; validation croisée BC03 réécrite pour être insensible aux versions | **Traité** |

## 6. Coûts et bénéfices (ROI)

**Coûts**

- **Charge** : ~4 semaines-personne (1 personne).
- **Infrastructure** : **0 €/mois** (sources publiques gratuites, exécution locale). Un déploiement
  cloud minimal coûterait ~0–7 €/mois sur une offre d'entrée de gamme.

**Bénéfices**

- **Réutilisabilité** : la chaîne (acquisition → ETL → grille → modèle → API) est générique ;
  changer d'espèces ou de région ne demande que d'ajuster `commun/config.py`.
- **Base d'un service opérationnel** : une association de suivi ornithologique (type LPO) pourrait
  s'en servir pour **prioriser les sorties de comptage** vers les semaines et zones à forte
  probabilité — gain de temps bénévole estimé à 1–2 sorties évitées par mois et par observateur.
- **Compétences démontrées** : les 6 blocs du référentiel, de l'infrastructure à l'industrialisation.

**ROI qualitatif** : coût d'infrastructure quasi nul, effort modéré, socle transposable à d'autres
problématiques de prévision spatio-temporelle sur données publiques.

## 7. Gouvernance des données & RGPD

- **RGPD** : le projet ne traite **aucune donnée à caractère personnel** (occurrences d'espèces,
  mesures météo). Détail des sources, licences et minimisation dans
  [`bc01_infrastructure_donnees/docs/architecture.md`](../bc01_infrastructure_donnees/docs/architecture.md).
- **Traçabilité** : les URL des API sont dans le code (`acquisition.py`) ; les jeux intermédiaires
  (`donnees/traitees/`) et le modèle de production (`modeles/pipeline_ml.pkl`) sont figés et
  versionnés.
- **Reproductibilité** : graines aléatoires fixées (`RANDOM_STATE`), hyperparamètres centralisés
  dans `commun/config.py`, entraînements suivis dans **MLflow** (`mlruns/`).

## Récapitulatif — ce qu'il faut retenir

BC06 ne produit pas d'algorithme : il **démontre le pilotage** du projet.

1. Le besoin métier a été **traduit** en un problème de classification binaire (point 1).
2. Le projet a suivi un **rétroplanning agile** de 4 semaines avec des jalons de contrôle (point 2).
3. Une suite de **tests automatisés** (rejoués ci-dessus, 6/6) fiabilise le code (point 3).
4. Chaque bloc est **documenté** au même format + un notebook (point 4).
5. Les **risques** sont nommés, la plupart traités ; les limites restantes sont **assumées**
   (points 5).
6. Le **coût d'infrastructure est nul**, le socle est **réutilisable** (point 6).
7. Le projet est **hors périmètre RGPD** et **reproductible** (point 7).

**Bilan du projet.** Les 6 blocs de compétences sont couverts : de l'acquisition des données
(BC01) à l'industrialisation (BC05), en passant par l'analyse exploratoire (BC02), le Machine
Learning (BC03) et le Deep Learning (BC04). Limites principales assumées : rappel des présences
perfectible (déséquilibre des classes), météo seule peu prédictive, déploiement cloud documenté
mais non réalisé.

Le script [`run.py`](run.py) rejoue les tests et affiche le planning et les limites en une commande.